# 🎮 Game QoE Intelligence Dashboard (2026)
### Professional Analysis Suite for Game Quality of Experience
---
**Project Scope:** This notebook serves as the visualization and insight layer for the QoE Analysis engine. It transforms raw network captures and visual labels into actionable research data.

#### 📑 Table of Contents:
1. **[Setup](#setup):** Environment configuration and parameters.
2. **[Data Bridge](#bridge):** Dynamic loading of pre-processed backend results.
3. **[Ground Truth](#truth):** Visual QoE validation and distribution.
4. **[Network Discovery](#network):** Protocol signatures and traffic stability.
5. **[Feature Engineering](#features):** Label distribution and cleaning.
6. **[Correlation Analysis](#correlation):** Statistical relationships between network metrics.
7. **[Signature Visualization](#signatures):** Distribution and Density (KDE) analysis.

In [ ]:
# 1 
# --- Global Execution Timer Start ---
from datetime import datetime # ניהול פורמטים של תאריכים ושעות
import time                   # מדידת זמנים ויצירת חותמות זמן (Timestamps) לשמות קבצים
# שמירת זמן ההתחלה של כל המחברת
notebook_start_time = time.time()
print(f"🚀 Overall Execution Started at: {datetime.now().strftime('%H:%M:%S')}")
print("-" * 40)

# קודם נייבא את הספריות האלה כדי שלא תהיה לנו בעיה עם הנתיבים בהמשך המחברת.
import os      # עבודה עם פקודות מערכת והגדרת נתיבים
import sys     # גישה למשתני מערכת וניהול נתיבי ייבוא (Python Path)
from pathlib import Path      # ניווט וניהול נתיבי קבצים בצורה מודרנית ובטוחה
# --- Michael: ספריות חדשות לניהול תהליכים במקביל (Multi-processing) ---
import multiprocessing # גישה למשאבי המעבד (CPU) ובדיקת כמות הליבות
from concurrent.futures import ProcessPoolExecutor # הכלי המרכזי לניהול הרצה מקבילית של משימות


'''
תא זה מרכז את כל הפרמטרים המשתנים של הפרויקט.
שינוי הערכים כאן ישפיע על כל המחברת - מתאים למעבר בין משחקים וסטודנטים שונים.
'''

# --- הגדרות רגישות להרצה במקביל ---
USE_PARALLEL = False     # מתג לבלוקי I/O (10, 18) - כבוי כברירת מחדל
USE_PARALLEL_ML = True   # מתג לבלוקי חישוב (34) - דלוק כברירת מחדל
CORES_TO_USE = max(1, multiprocessing.cpu_count() - 4) # חישוב כמות ליבות זמינה להרצה מקבילית - משאירים 4 ליבות פנויות כדי לא להקריס את המערכת

INCLUDE_SVM = False   # מתג למודל ה-SVM: False לדילוג (חוסך שעות ריצה), True להכללה בניתוח הסופי

# 1. ניהול נתיבים (Path Management)
# פונקציית getcwd מחזירה את המיקום של הקובץ הנוכחי כדי שכל הנתיבים יהיו יחסיים ולא מוחלטים
CURRENT_DIR = Path(os.getcwd())
# BASE_DIR מצביע על תיקיית האב (ping_extractor) כדי לאפשר גישה לקבצי עזר בתיקיות מקבילות
BASE_DIR = CURRENT_DIR.parent

# הגדרת תיקיית המשנה של הדאטה הנוכחי - זהו המשתנה המרכזי למעבר בין ניסויים שונים
DATA_SUBFOLDER = "P++200" 

# הגדרת נתיב המקור המלא לקבצי ה-PCAP והתמונות בתוך מבנה תיקיות ה-adb
RAW_DATA_PATH = BASE_DIR.parent / "adb" / "sessions_screencap_pc_roi" / DATA_SUBFOLDER

# יצירת נתיב ייעודי לתוצרים (גרפים, אקסלים ו-CSV) עבור כל ניסוי בנפרד
PRODUCTS_DIR = CURRENT_DIR / f"Products_{DATA_SUBFOLDER}"
# יצירת התיקייה פיזית על הדיסק במידה והיא עדיין לא קיימת
os.makedirs(PRODUCTS_DIR, exist_ok=True)

# 2. הגדרות רשת ופורטים (Network & Port Settings)
# טווח הפורטים שבו המשחק משתמש - חיוני לסינון רעשי רקע (כמו דפדפן או עדכוני מערכת)
GAME_PORT_RANGE = "10010-10025" 
# יצירת פילטר BPF (Berkeley Packet Filter) שמועבר ישירות לכרטיס הרשת דרך NFStreamer
BPF_FILTER = f"udp and (portrange {GAME_PORT_RANGE})"

# 3. פרמטרי ניתוח (Analysis Hyper-parameters)
WINDOW_SIZE = 10     # גודל חלון הזיכרון (בשניות) עבור חישובי ממוצע וסטטיסטיקה רצה
DECAY_RATE = 0.5     # קצב הירידה בחשיבות של דאטה ישן בתוך החלון (Weighted Decay)
SAMPLE_DURATION = 1  # רזולוציית הזמן - הקוד "יחתוך" את תעבורת הרשת למקטעים של שנייה אחת

# הוספת BASE_DIR ל-sys.path: פעולה קריטית המאפשרת לפייתון למצוא את הקבצים rank.py ו-modelse.py 
# הנמצאים בתיקייה אחת מעלה, ללא קשר למיקום שבו המחברת רצה.
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

# 4. שמות קבצי התוצרים (File Name Configuration)
# הגדרת שמות הקבצים במקום אחד לשינוי קל בעתיד
METADATA_FILENAME = "metadata_results_auto.csv"
NETWORK_ANALYSIS_FILENAME = "NetDiscovery_Michael_gm.xlsx"
FEATURES_FILENAME = "final_training_features.csv"
IMPORTANCE_FILENAME = "feature_importance_results.csv"

# הדפסת סיכום הגדרות למשתמש לאימות לפני תחילת הריצה הכבדה
print(f"--- Environment Configured ---")
print(f"Working Directory: {CURRENT_DIR}")
print(f"Data Source: {RAW_DATA_PATH}")
print(f"Network Filter: {BPF_FILTER}")
print(f"Parallel Mode (I/O - Blocks 10,18): {USE_PARALLEL}")
print(f"Parallel Mode (ML - Block 34): {USE_PARALLEL_ML} (Using {CORES_TO_USE} cores)")

<a id="setup"></a>
## 🛠️ Step 1: Environment & Dependency Setup
In this section, we initialize the paths for **P++200** and import the specialized libraries for network analysis (`NFStreamer`) and machine learning (`scikit-learn`).

In [ ]:
# 2
'''
ריכוז כל הספריות הנדרשות למחברת במקום אחד.
הסדר מבטיח שכל הכלים (כולל openpyxl, nfstream ו-sklearn) יהיו זמינים לכל אורך הניתוח.
'''

# ספריות ליבה של מערכת ההפעלה וניהול קבצים

import pandas as pd  # הספרייה המרכזית לניהול וניתוח טבלאות נתונים (DataFrames)
import numpy as np   # חישובים מתמטיים מתקדמים ועבודה עם מערכים רב-ממדיים
import matplotlib.pyplot as plt # יצירת גרפים ותרשימים בסיסיים
import seaborn as sns           # ספריית ויזואליזציה מתקדמת המבוססת על Matplotlib
import re      # עבודה עם ביטויים רגולריים (Regular Expressions) לחיפוש וסינון טקסט
import csv     # קריאה וכתיבה של קבצי CSV
import ast     # ניתוח מבני נתונים (כמו הפיכת מחרוזת מילוי למילון פייתון)
import json    # עבודה עם פורמט נתונים JSON
import uuid    # יצירת מזהים ייחודיים (Unique Identifiers) במידת הצורך

# ספריות ייעודיות לניתוח תעבורת רשת והפקת דוחות אקסל מעוצבים
from nfstream import NFStreamer # הכלי המרכזי לניתוח קבצי PCAP והמרתם לזרימות רשת (Flows)
from openpyxl import Workbook    # יצירת קבצי Excel מאפס
from openpyxl.styles import PatternFill, Font, Alignment # עיצוב תאים, גופנים ויישור טקסט באקסל

# ספריות Machine Learning לביצוע חיזויים וניתוח חשיבות פיצ'רים
from sklearn.model_selection import train_test_split # חלוקת הדאטה לסט אימון וסט בדיקה
from sklearn.inspection import permutation_importance # כלי לניתוח תרומת כל פיצ'ר לדיוק המודל
from joblib import Parallel, delayed # Michael: כלי חזק ויציב יותר להרצה במקביל ב-Windows

# ייבוא מודולים מקומיים (קבצי .py שנמצאים בתיקיית הפרויקט)
# rank: פונקציות לדירוג איכות הפינג (QoE)
# modelse: פונקציות לבנייה והרצה של מודלים שונים
from rank import rank_qoe_3_classes, rank_str_to_int_mapping_3_classes
from modelse import make_models

# הגדרות תצוגה גלובליות עבור הגרפים במחברת
%matplotlib inline
sns.set_theme(style="whitegrid") # הגדרת רקע לבן עם רשת לגרפים של Seaborn

print("✅ All modules imported successfully.")

## 📋 Step 2: Model Feature Definition
Definition of the "Toolbox" — the full list of raw and statistical network features used for training our predictive models.

In [ ]:
# 3
'''
בלוק זה מגדיר את "ארגז הכלים" של המודלים - רשימת כל הפיצ'רים (תכונות) שישמשו לאימון.
הרשימה כוללת מדדי רשת גולמיים לצד מדדים סטטיסטיים שחושבו על פני חלון הזמן בבלוק 018.
'''

FEATURE_COLUMNS = [
            # --- תכונות גודל פקטה: כיוון שליחה מהמכשיר לשרת (Upload / Src2Dst) ---
            'src2dst_avg_pkt_size',        # הערך הגולמי של גודל הפקטה הממוצע בשנייה הנוכחית
            'src2dst_avg_pkt_size_mean',   # ממוצע גודל פקטה משוקלל עם דעיכה (על פני 10 שניות)
            'src2dst_avg_pkt_size_median', # חציון גודל הפקטה בחלון הזמן
            'src2dst_avg_pkt_size_std',    # סטיית תקן - מדד לתנודתיות ושינויים בגודל החבילות הנשלחות
            'src2dst_avg_pkt_size_min',    # הגודל המינימלי שנמדד בחלון
            'src2dst_avg_pkt_size_max',    # הגודל המקסימלי שנמדד בחלון (מזהה חבילות מידע גדולות וחריגות)

            # --- תכונות PPS (חבילות לשנייה): כיוון שליחה (Upload / Src2Dst) ---
            'src2dst_pps',        # קצב שליחת חבילות גולמי בשנייה האחרונה
            'src2dst_pps_mean',   # ממוצע PPS משוקלל עם דעיכה (נותן עדיפות למידע העדכני ביותר)
            'src2dst_pps_median', # חציון קצב השליחה ב-10 השניות האחרונות
            'src2dst_pps_std',    # מדד ליציבות קצב השליחה (Jitter בשכבת היישום)
            'src2dst_pps_min',    # קצב מינימלי שנרשם בחלון הזמן
            'src2dst_pps_max',    # קצב מקסימלי שנרשם (מזהה קפיצות עומס/Burstiness)

            # --- תכונות גודל פקטה: כיוון קבלה מהשרת למכשיר (Download / Dst2Src) ---
            'dst2src_avg_pkt_size',        # הערך הגולמי של גודל הפקטה בשנייה הנוכחית
            'dst2src_avg_pkt_size_mean',   # ממוצע משוקלל עם דעיכה של גודל הפקטה הנכנס
            'dst2src_avg_pkt_size_median', # חציון גודל הפקטה הנכנס
            'dst2src_avg_pkt_size_std',    # תנודתיות בגודל חבילות המידע הנכנסות מהשרת
            'dst2src_avg_pkt_size_min',    # גודל מינימלי נכנס
            'dst2src_avg_pkt_size_max',    # גודל מקסימלי נכנס

            # --- תכונות PPS (חבילות לשנייה): כיוון קבלה (Download / Dst2Src) ---
            'dst2src_pps',        # קצב קבלת חבילות גולמי בשנייה הנוכחית
            'dst2src_pps_mean',   # ממוצע PPS נכנס משוקלל (קריטי לזיהוי רצף נתונים מהשרת)
            'dst2src_pps_median', # חציון קצב הקבלה
            'dst2src_pps_std',    # מדד ליציבות זרם הנתונים הנכנס מהשרת
            'dst2src_pps_min',    # קצב קבלה מינימלי
            'dst2src_pps_max',    # קצב קבלה מקסימלי

            # --- תכונות יחס כיווניות (Direction Ratio) ---
            # מדד זה בוחן את האיזון בין העלאה להורדה, נתון חשוב בזיהוי פרוטוקולי משחק.
            'pkt_dir_ratio',        # היחס בין חבילות יוצאות לנכנסות בשנייה הנוכחית
            'pkt_dir_ratio_mean',   # ממוצע יחס הכיווניות המשוקלל לאורך 10 שניות
            'pkt_dir_ratio_median', # חציון יחס הכיווניות
            'pkt_dir_ratio_std',    # מדד ליציבות היחס בין כיווני התנועה
            'pkt_dir_ratio_min',    # הערך המינימלי של היחס בחלון
            'pkt_dir_ratio_max'     # הערך המקסימלי של היחס בחלון
]

<a id="bridge"></a>
## 🌉 Step 3: The Smart Data Loading Bridge
This critical block automatically detects the latest archive generated by the Backend engine. It ensures the notebook is always working with the most recent results without re-processing hours of data.

In [ ]:
# 04 
# --- Smart Data Loading Bridge (Full Version) ---
# בלוק זה משמש כ"גשר" (Bridge) בין מנוע ה-Backend לבין המחברת. 
# מטרתו העיקרית: טעינת התוצאות המוכנות מתיקיית ה-Products כדי לחסוך עיבוד מחדש שלוקח שעות.

print(f"📥 Loading analysis results from: {PRODUCTS_DIR}")

# 1. טעינה חכמה של קובץ הפיצ'רים (training_df)
# המערכת מחפשת את כל הקבצים שמתחילים ב-"window_10_" ומסתיימים ב-CSV (קבצי הארכיון שנוצרו ב-Backend)
archive_files = list(PRODUCTS_DIR.glob("window_10_*.csv"))

if archive_files:
    # לוגיקת בחירת הקובץ:
    # אנו ממיינים את הקבצים לפי שני פרמטרים: (1) זמן השינוי האחרון - st_mtime, ו-(2) גודל הקובץ - st_size.
    # reverse=True דואג שהקובץ הכי חדש (והכי גדול במידה והזמנים זהים) יהיה ראשון ברשימה [0].
    latest_file = sorted(archive_files, key=lambda x: (x.stat().st_mtime, x.stat().st_size), reverse=True)[0]
    
    # טעינת נתוני האימון מהארכיון שנמצא
    training_df = pd.read_csv(latest_file)
    
    # יצירת משתנה עבודה מקוצר 'df' - זהו הכינוי שבו משתמשים רוב בלוקי הויזואליזציה במחברת
    df = training_df 
    print(f"✅ Loaded latest features archive: {latest_file.name} ({len(df)} samples)")
else:
    # תוכנית גיבוי: אם לא נמצא קובץ עם חותמת זמן, ננסה לטעון את קובץ ברירת המחדל
    default_features_path = PRODUCTS_DIR / FEATURES_FILENAME
    if default_features_path.exists():
        training_df = pd.read_csv(default_features_path)
        df = training_df
        print(f"✅ Loaded default features file: {FEATURES_FILENAME}")
    else:
        # התראה למשתמש במקרה שתיקיית התוצרים ריקה לחלוטין
        print(f"⚠️ Warning: {FEATURES_FILENAME} not found!")


# 2. טעינת המטא-דאטה (metadata)
# קובץ זה מכיל את הקישור בין התיקיות, קבצי ה-PCAP וערכי הפינג המקוריים שחולצו
metadata_path = PRODUCTS_DIR / METADATA_FILENAME
if metadata_path.exists():
    metadata = pd.read_csv(metadata_path)
    print(f"✅ Metadata loaded: {len(metadata)} runs.")
else:
    print(f"⚠️ Warning: {METADATA_FILENAME} not found!")

# 3. טעינת ניתוח הרשת המדויק (net_df)
# זהו קובץ ה-Excel שנוצר על ידי שילוב Scapy ו-NFStreamer. 
# הוא מכיל את מדדי ה"טוהר" (Purity), ה-UDP% וה-IAT שמאמתים את איכות הדאטה.
net_path = PRODUCTS_DIR / NETWORK_ANALYSIS_FILENAME
if net_path.exists():
    # שים לב: השימוש ב-read_excel הכרחי כאן כי הקובץ נשמר בפורמט אקסל מעוצב
    net_df = pd.read_excel(net_path)
    print(f"✅ Network Analysis loaded: {len(net_df)} sessions.")
else:
    print(f"⚠️ Warning: {NETWORK_ANALYSIS_FILENAME} not found!")

# 4. טעינת חשיבות הפיצ'רים (imp_df)
# קובץ זה מרכז את תוצאות ה-Permutation Importance מכל שבעת מודלי ה-ML שהורצו ב-Backend.
# הוא משמש ליצירת גרף הקונצנזוס הצבעוני בסוף המחברת.
imp_path = PRODUCTS_DIR / IMPORTANCE_FILENAME
if imp_path.exists():
    imp_df = pd.read_csv(imp_path)
    print(f"✅ Feature Importance loaded.")
else:
    print(f"⚠️ Warning: {IMPORTANCE_FILENAME} not found!")

<a id="truth"></a>
## 👁️ Step 4: Ground Truth & Visual Validation
Synchronizing the visual data (ping from screenshots) with the timeline to ensure our labels are accurate before analysis.

In [ ]:
# 5
# --- Helper functions for visualization ---
# בלוק זה מרכז פונקציות עזר שמטרתן להפוך את המידע הגולמי (שמות קבצים וטקסט) 
# לנתונים מספריים וטבלאות שניתן לצייר עליהן גרפים של ציר זמן.

def hhmmss_to_seconds(hhmmss: int) -> int:
    """
    ממירה פורמט זמן של מספר שלם (HHMMSS) למספר שניות כולל מתחילת היממה.
    למשל: המספר 103005 (עשר וחצי וחמש שניות) יהפוך ל-37,805 שניות.
    """
    # חילוץ השעות: חילוק ב-10,000 נותן את שתי הספרות השמאליות
    h = hhmmss // 10000 
    # חילוץ הדקות: חילוק ב-100 נותן את השעות והדקות, ואז שארית החילוק ב-100 מבודדת את הדקות
    m = (hhmmss // 100) % 100
    # חילוץ השניות: השארית מחילוק ב-100 נותנת את שתי הספרות הימניות ביותר
    s = hhmmss % 100
    # חישוב סך השניות: שעה = 3600 שניות, דקה = 60 שניות
    return h * 3600 + m * 60 + s

def to_qoe_class_zero(ping):
    """
    הופכת ערך פינג גולמי למספר מחלקה (1, 2, או 3) לפי איכות החוויה.
    אם הנתון פגום או חסר, היא מחזירה 0 כדי לא לשבש את הסטטיסטיקה.
    """
    # בדיקה ראשונית: האם הערך הוא NaN (נתון חסר ב-Pandas)
    if pd.isna(ping): return 0
    try: 
        # ניסיון להפוך את הערך למספר עשרוני (למקרה שהפינג הגיע כמחרוזת טקסט)
        ping = float(ping)
    except Exception: 
        # אם יש טקסט שלא ניתן להמיר למספר (כמו "Error"), מחזירים 0
        return 0
    # קריאה לפונקציה החיצונית rank_qoe_3_classes לקבלת תווית (Good/Fair/Poor)
    label = rank_qoe_3_classes(ping)
    # המרת תווית הטקסט למספר (1, 2, 3) בעזרת מילון המיפוי; מחזיר 0 אם לא נמצא
    return rank_str_to_int_mapping_3_classes.get(label, 0)

def build_series_from_row(row):
    """
    בונה טבלת נתונים (DataFrame) עבור סשן בודד, המציגה את רמת ה-QoE לאורך זמן.
    הפונקציה מסנכרנת בין רגע תחילת המשחק לבין כל צילום מסך שנלקח.
    """
    # חילוץ שעת ההתחלה משם התיקייה (למשל run_..._123000) והמרה לשניות
    start_hhmmss = int(row["folder"].split("_")[-1])
    start_sec = hhmmss_to_seconds(start_hhmmss)
    
    # הפיכת מחרוזת הטקסט המייצגת מילון (time_value_pairs) לאובייקט מילון פייתון אמיתי
    pairs = ast.literal_eval(row["time_value_pairs"])
    
    xs, ys = [], [] # רשימות לציר ה-X (זמן) וציר ה-Y (איכות)
    
    # מעבר על כל זוג של "זמן צילום : ערך פינג" בתוך המילון
    for t_str, ping in pairs.items():
        # חישוב ה-Offset: כמה שניות עברו מרגע תחילת הסשן ועד רגע צילום הפינג הזה
        dt = hhmmss_to_seconds(int(t_str)) - start_sec
        
        # סינון רעשים: אם זמן הצילום קטן מזמן התחלת התיקייה, זה נתון לא תקין - מדלגים
        if dt < 0: continue
        
        xs.append(dt) # הוספת השנייה היחסית לרשימת הזמנים
        ys.append(to_qoe_class_zero(ping)) # הוספת דרגת ה-QoE המחושבת לרשימת הערכים
        
    # החזרת טבלת Pandas מסודרת וממוינת לפי ציר הזמן, כדי שהגרף ייראה כקו רציף
    return pd.DataFrame({"t_sec": xs, "qoe": ys}).sort_values("t_sec")

In [ ]:
# 6
'''
תא זה מציג ויזואליזציה של ה-QoE לאורך זמן עבור סשן ספציפי.
'''

# בחירת שורה אקראית (במקרה זה אינדקס 12) מתוך טבלת המטא-דאטה שנוצרה בזיכרון
row0 = metadata.iloc[12]

# שימוש בפונקציית העזר מבלוק 06 כדי לבנות ציר זמן של איכות החוויה (QoE) לסשן הנבחר
s0 = build_series_from_row(row0)

# יצירת "מסכה" לזיהוי רגעים שבהם הנתונים חסרים או לא תקינים (דירוג 0)
mask0 = (s0["qoe"] == 0)

# הגדרת ממדי הגרף (רוחב וגובה) לתצוגה ברורה של ציר הזמן
plt.figure(figsize=(12,4))

# קו כחול - ייצוג ויזואלי של ה-QoE התקין (דירוגים 1, 2, 3)
# הערה: אנו מחליפים ערכי 0 ב-NaN כדי שהקו בגרף ייקטע בנקודות שבהן אין מידע
qoe_line = s0["qoe"].where(~mask0, np.nan)
plt.plot(s0["t_sec"], qoe_line, marker="o", linestyle="-", label="qoe (1/2/3)")

# נקודות אדומות - סימון מפורש של נקודות בזמן שבהן ה-QoE=0 (מידע חסר)
# המטרה: לאפשר לחוקר לזהות "חורים" ברצף הנתונים הוויזואליים
plt.scatter(s0.loc[mask0, "t_sec"], s0.loc[mask0, "qoe"],
            color="red", zorder=5, label="qoe=0")

# הגדרות צירים וכותרות
plt.yticks([0,1,2,3])                        # הצגת שנתות עבור דרגות האיכות בלבד
plt.ylim(-0.5, 3.5)                          # קביעת גבולות הציר האנכי למניעת חיתוך הנקודות
plt.xlabel("Time (seconds from run start)")  # ציר X מייצג שניות מתחילת המשחק
plt.ylabel("QoE class (0/1/2/3)")            # ציר Y מייצג את רמת חווית המשתמש
plt.title(f"QoE over time - {row0['folder']} / {row0['pcapng_file']}") # כותרת דינמית לפי הסשן הנבחר
plt.grid(True)                               # הוספת רשת למדידה נוחה של הזמנים
plt.legend()                                 # הצגת מקרא הגרף
plt.show()                                   # הצגת התוצאה הסופית על המסך

In [ ]:
# 7
'''
תא זה מחשב סטטיסטיקות מסכמות עבור מחלקות ה-QoE על פני כל בסיס הנתונים.
המטרה: להבין את התפלגות הנתונים (Data Distribution) ולוודא שהמודלים יתאמנו על דאטה מאוזן ככל הניתן.
הוא משתמש במשתנה metadata הגלובלי לצורך החישובים המצטברים.
'''

def sume_ech_class(row):
    '''
    פונקציית עזר הממיינת את כל הפינגים של סשן בודד לשלוש רמות האיכות.
    היא מחזירה שלוש רשימות המכילות את ערכי הפינג המקוריים לכל מחלקה.
    '''
    folder = row["folder"]
    # המרת ייצוג הטקסט של הפינגים לאובייקט מילון פייתון פעיל
    pairs = ast.literal_eval(row["time_value_pairs"])
    class_1, class_2, class_3 = [], [], []
    
    # מעבר על כל הפינגים שנאספו בסשן הנוכחי
    for _t_str, ping in pairs.items():
        # סינון ערכים ריקים (None) כדי לא להטות את הסטטיסטיקה
        if ping is None:
            continue
        
        # המרת ערך הפינג למספר המחלקה (1-3) בעזרת הפונקציה מבלוק 06
        q = to_qoe_class_zero(ping)
        
        # שיוך ערך הפינג לרשימה המתאימה לפי דרגת האיכות שלו
        if q == 1: class_1.append(ping)
        elif q == 2: class_2.append(ping)
        elif q == 3: class_3.append(ping)
        
    return class_1, class_2, class_3

# שלב צבירת הנתונים: יצירת רשימות שיכילו את כמות המופעים בכל סשן
all_c1, all_c2, all_c3 = [], [], []

# לולאה הרצה על כלל הסשנים המופיעים בטבלת המטא-דאטה (כ-400 משחקים ב-P++300)
for i in metadata.index:
    row = metadata.loc[i]
    # חילוץ הרשימות עבור הסשן הנוכחי
    c1, c2, c3 = sume_ech_class(row)
    
    # שמירת אורך הרשימה (כמות הדגימות) עבור כל מחלקה בסשן הזה
    all_c1.append(len(c1))
    all_c2.append(len(c2))
    all_c3.append(len(c3))

# הדפסת התוצאות הסופיות למסך:
# 1. הסכום הכולל (sum) - כמה שניות של דאטה יש לנו מכל רמת איכות בכל המחקר.
print(f"Class 1 count: {sum(all_c1)}, Class 2 count: {sum(all_c2)}, Class 3 count: {sum(all_c3)}")

# 2. סטיית התקן (std) - מראה עד כמה כמות הדגימות משתנה בין סשנים שונים (מעיד על עקביות הנתונים).
print(f"Class 1 std: {np.std(all_c1)}, Class 2 std: {np.std(all_c2)}, Class 3 std: {np.std(all_c3)}")

<a id="network"></a>
## 🌐 Step 5: Network Discovery & Protocol Validation
Quantitative proof that the captured traffic is game-related (UDP dominance) and validation of connection density across 302 sessions.

In [ ]:
# 8
''' תא זה מציג את התפלגות הפרוטוקולים (UDP מול TCP). הוא משתמש ישירות ב-net_df שנוצר בתא הקודם. '''
# אין צורך בבדיקות if או בטעינת CSV - המידע כבר בזיכרון ב-net_df

# חישוב הערך הממוצע של אחוז תעבורת ה-UDP מכלל הסשנים שנותחו
udp_avg = net_df['udp_pct'].mean()
# חישוב הערך הממוצע של אחוז תעבורת ה-TCP מכלל הסשנים שנותחו
tcp_avg = net_df['tcp_pct'].mean()

# הגדרת גודל שטח הציור (רוחב וגובה באינצ'ים)
plt.figure(figsize=(8, 6))

# הגדרת צבעים ייחודיים: ירוק לתעבורת משחק (UDP) וצהוב לתעבורת בקרה/ניהול (TCP)
colors = ['#4CAF50', '#FFC107'] # Green for UDP (Game), Yellow for TCP

# יצירת גרף עוגה (Pie Chart) להצגת ההתפלגות הפרוטוקולרית:
# labels - תוויות עבור כל חלק בעוגה
# autopct - הצגת האחוזים על גבי הגרף בדיוק של ספרה אחת אחרי הנקודה
# startangle - סיבוב הגרף ב-140 מעלות לתצוגה אסתטית יותר
# explode - הפרדת החלק של ה-UDP משאר העוגה כדי להדגיש אותו כמרכזי במחקר
# shadow - הוספת אפקט הצללה לגרף למראה תלת-ממדי
plt.pie([udp_avg, tcp_avg], labels=['UDP (Game Traffic)', 'TCP'], 
        autopct='%1.1f%%', startangle=140, explode=(0.1, 0), shadow=True)

# הוספת כותרת לגרף המדגישה את מטרת האימות (ווידוא שתפסנו תעבורת משחק אמיתית)
plt.title('Overall Network Protocol Distribution\n(Validation of Game Traffic Capture)')

# הצגת הגרף המוגמר
plt.show()

# הדפסת מספר הסשנים הכולל ששימשו לחישוב הממוצעים המוצגים בגרף
print(f"Total sessions analyzed: {len(net_df)}")

In [ ]:
# 9
'''
תא זה מציג ניתוח של ה-IAT (Jitter) וגודל הפקטות הממוצע.
מדדים אלו קריטיים להבנת היציבות של רשת המשחק וזיהוי חתימת התעבורה הייחודית שלו.
הוא משתמש בנתונים הסטטיסטיים שנאספו בבלוק 010 עבור כל סשן משחק בנפרד.
'''
# הגדרת משטח הציור (Figure) הכולל שני גרפים המוצגים זה לצד זה (סאב-פלוטים)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# גרף 1: התפלגות ה-IAT (Network Jitter)
# ה-IAT (Inter-Arrival Time) מייצג את הזמן שעובר בין הגעת חבילות מידע עוקבות.
# שונות גבוהה בערך זה מעידה על Jitter, גורם מרכזי שפוגע בחוויית המשחק בזמן אמת.
# יצירת היסטוגרמה שמראה כמה סשנים נמצאים בכל טווח של זמני הגעה (IAT)
ax1.hist(net_df['avg_iat_ms'], bins=20, color='skyblue', edgecolor='black', alpha=0.7)
# הוספת קו אנכי אדום המציין את החציון (Median) - הערך המייצג של ה-Jitter בכלל הניסוי
ax1.axvline(net_df['avg_iat_ms'].median(), color='red', linestyle='dashed', linewidth=1.5, label='Median IAT')
ax1.set_title('Network Jitter Distribution\n(Average IAT per Session)')
ax1.set_xlabel('Mean Inter-Arrival Time (ms)')
ax1.set_ylabel('Frequency (Number of Sessions)')
ax1.legend()

# גרף 2: חתימת גודל הפקטה (Packet Size Signature)
# גרף פיזור (Scatter Plot) המציג את הגודל הממוצע של חבילות המידע בכל סשן בנפרד.
# עקביות בערכים אלו לאורך מאות הסשנים מאמתת שמדובר באפליקציה עם אופי תעבורה קבוע (המשחק הנבדק).
ax2.scatter(net_df.index, net_df['avg_pkt_size'], alpha=0.6, color='purple', s=40)
ax2.set_title('Mean Packet Size per Session\n(Consistency Validation)')
ax2.set_xlabel('Session Index')
ax2.set_ylabel('Average Packet Size (Bytes)')
# הוספת רשת (Grid) כדי לאפשר קריאה מדויקת של ערכי גודל הפקטה
ax2.grid(True, linestyle='--', alpha=0.5)

# התאמה אוטומטית של המרווחים בין הגרפים למניעת חפיפה בין הכותרות
plt.tight_layout()
# הצגת התוצאה הוויזואלית הסופית על המסך
plt.show()

In [ ]:
# 10
# ניסיון למצוא את הדאטה בזיכרון או לטעון מהקובץ
'''
תא זה מנתח את "צפיפות החיבורים" (Flow Density) בכל סשן.
מטרת הבדיקה: לוודא שפרופיל הקישוריות של המשחק עקבי לאורך כל מאות הניסויים,
ולזהות האם קיימת "רעש" רקע חריג (כמו עדכוני מערכת או שירותים אחרים) שתפסנו בטעות.
'''

xlsx_name = NETWORK_ANALYSIS_FILENAME
# מנגנון הגנה: בדיקה האם המשתנה net_df כבר קיים בזיכרון (מהרצה קודמת של בלוק 010)
if 'net_df' in locals():
    plot_df = net_df
# אם המשתנה לא בזיכרון, ניסיון לטעון את נתוני ניתוח הרשת מתוך קובץ ה-CSV בתיקיית התוצרים
elif (PRODUCTS_DIR / xlsx_name).exists():
    plot_df = pd.read_excel(PRODUCTS_DIR / xlsx_name)
else:
    # הודעת שגיאה במקרה ששני המקורות לא זמינים
    plot_df = None
    print(f"Error: Could not find '{xlsx_name}' in the folder. Please ensure the file exists.")

if plot_df is not None:
    # הגדרת ממדי הגרף
    plt.figure(figsize=(12, 6))
    
    plt.scatter(plot_df.index, plot_df['total_flows'], alpha=0.6, color='dodgerblue', s=50, edgecolors='k')
    
    # חישוב והוספת קו ממוצע (Baseline)
    avg_flows = plot_df['total_flows'].mean()
    # יצירת קו אופקי (axhline) מקווקו באדום כנקודת ייחוס (Baseline) ליציבות המערכת
    plt.axhline(avg_flows, color='red', linestyle='--', linewidth=2, label=f'Average Flows: {avg_flows:.1f}')
    
    # הגדרות כותרות וצירים למחקר
    plt.title('Network Flow Density per Session\n(Connectivity Profile Validation)', fontsize=14, pad=15)
    plt.xlabel('Session Index', fontsize=12) # אינדקס הסשנים לאורך זמן    
    plt.ylabel('Number of Unique Flows', fontsize=12) # כמות החיבורים הייחודיים שנוצרו (TCP/UDP)
    plt.legend(loc='upper right') # הצגת המקרא בפינה הימנית העליונה
    plt.grid(True, linestyle='--', alpha=0.5) # הוספת רשת עדינה למדידה נוחה של הערכים
    
    # סידור אוטומטי של האלמנטים בגרף והצגתו
    plt.tight_layout()
    plt.show()

<a id="features"></a>
## 🧹 Step 6: Feature Cleaning & Label Distribution
Preparing the dataset for Machine Learning by removing incomplete samples and calculating the target class distribution.

In [ ]:
# 11
'''
תא זה מתמקד בניקוי סופי של הנתונים והכנת עמודת המטרה (Target Label) עבור המודלים.
שלב זה קריטי כדי להבטיח שהמודל לא ינסה ללמוד מנתונים חסרים או משובשים.
'''

# העברת נתוני האימון שנוצרו בבלוק 018 למשתנה עבודה מקוצר (df) לצורך ניתוח והצגה
df = training_df

# ניקוי ראשון: הסרת כל השורות שבהן אין ערך פינג (Label חסר)
# שורות אלו חסרות ערך ללמידה מאולצת (Supervised Learning) כי אין למודל "תשובה" להשוות אליה.
df = df.dropna(subset=['ping'])

# ניקוי שני: הסרת שורות שבהן חסרים נתוני רשת (Features) באחת העמודות שהגדרנו בבלוק 019
# פערים אלו עלולים להיווצר עקב מחיקת תמונות מהניסוי או חורים בתפיסת הפקטות.
df = df.dropna(subset=FEATURE_COLUMNS)

# יצירת עמודת המטרה 'y': המרת ערכי הפינג הגולמיים למספרים המייצגים את מחלקות האיכות (1, 2, 3)
# Michael: שימוש בשם המשתנה 'y' לצורך תאימות מלאה למודלים המקוריים של דביר ומחברות קודמות.

# טעינת המיפוי והגנה על ערכי 0 
df['y'] = df['ping'].apply(lambda p: rank_str_to_int_mapping_3_classes.get(rank_qoe_3_classes(p), 0))
df = df[df['y'] > 0].copy()

# ספירת כמות הדגימות המצטברת מכל סוג (איכות טובה, בינונית וגרועה)
counts = df['y'].value_counts()

# הגדרת ממדי הגרף להצגת התפלגות המחלקות
plt.figure(figsize=(6,6))

# יצירת גרף עוגה המראה את הפיזור של הנתונים בין המחלקות השונות (Label Distribution)
# autopct='%1.1f%%' - מציג את אחוז הדגימות בכל מחלקה על גבי הגרף
# startangle=140 - סיבוב אסתטי של הגרף לתצוגה ברורה
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140)

# הוספת כותרת המבהירה כי זהו ניתוח התפלגות הלייבלים במחקר
plt.title('Distribution by Label')

# הבטחת צורת עיגול מושלמת לגרף העוגה
plt.axis('equal')  

# הצגת הגרף המוגמר על המסך
plt.show()

<a id="correlation"></a>
## 📊 Step 7: Statistical Correlation Insights
Using **Spearman Correlation** to identify redundant features and discover which network metrics are the strongest "signals" for high-latency events.

In [ ]:
# 12
# חישוב מטריצת הקורלציה מסוג ספירמן
'''
תא זה מחשב ומציג את הקשרים (קורלציות) בין כל הפיצ'רים השונים שיצרנו.
המטרה: לזהות פיצ'רים דומים מדי (Multicollinearity) ולהבין אילו מדדי רשת מתנהגים בצורה דומה.
אנו משתמשים בשיטת 'ספירמן' (Spearman) כי היא מתאימה לזיהוי קשרים מגמתיים גם אם אינם ליניאריים מושלמים.
'''

# חישוב מטריצת הקורלציה על כל העמודות שהגדרנו ברשימת ה-FEATURE_COLUMNS
corr = df[FEATURE_COLUMNS].corr(method='spearman')

# הגדרת ממדי המפה (רוחב 14, גובה 12) כדי שכל שמות הפיצ'רים יהיו קריאים
plt.figure(figsize=(14, 12))

# יצירת מפת חום (Heatmap) ויזואלית בעזרת ספריית Seaborn:
# corr - המטריצה שחישבנו
# cmap='coolwarm' - סקאלת צבעים שבה כחול מייצג קשר שלילי, אדום קשר חיובי ולבן חוסר קשר
# center=0 - קביעת נקודת האפס כמרכז הצבעים (לבן)
# square=True - שמירה על תאים ריבועיים למראה מאורגן
# cbar_kws={'shrink': 0.8} - הקטנת מקרא הצבעים בצד כדי שיתאים לגובה הגרף
sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0,
    square=True,
    cbar_kws={'shrink': 0.8}
)

# הוספת כותרת לתיאור הניתוח
plt.title("Spearman Correlation Between Features")

# הצגת המפה הסופית על המסך
plt.show()

In [ ]:
# 13
'''
תא זה מבצע סינון אוטומטי של "זוגות פיצ'רים" בעלי מתאם (קורלציה) גבוה מאוד.
המטרה: לאתר יתירות (Redundancy) במידע. אם שני פיצ'רים מתואמים כמעט לחלוטין, 
הם מספקים למודל את אותו מידע בדיוק, וניתן לעיתים לוותר על אחד מהם כדי לפשט את המודל.
'''

# חישוב מטריצת המתאם בין כל המשתנים שנבחרו למחקר
corr_features = df[FEATURE_COLUMNS].corr(method='spearman')

# הגדרת סף (Threshold) לקורלציה גבוהה - 0.85 ומעלה נחשב לקשר חזק מאוד
threshold = 0.85
high_corr = []

# לולאה כפולה שעוברת על המשולש העליון של המטריצה כדי למצוא זוגות ייחודיים של משתנים
for i in range(len(corr_features.columns)):
    for j in range(i + 1, len(corr_features.columns)):
        # חילוץ ערך המתאם בין פיצ'ר i לפיצ'ר j
        val = corr_features.iloc[i, j]
        # בדיקה האם הערך המוחלט של המתאם חוצה את הסף שהגדרנו (חיובי או שלילי חזק)
        if abs(val) > threshold:
            # הוספת הזוג לרשימת הפיצ'רים ה"חשודים" כיתירים
            high_corr.append((
                corr_features.columns[i],
                corr_features.columns[j],
                val
            ))

# המרת רשימת הזוגות לטבלת נתונים (DataFrame) לצורך הצגה ומיון נוחים
high_corr_df = pd.DataFrame(high_corr, columns=['feature_1', 'feature_2', 'spearman_corr'])
# מיון הטבלה מהקורלציה הכי גבוהה (קרוב ל-1.0) ומטה כדי לראות קודם את היתירות המשמעותית ביותר
high_corr_df = high_corr_df.sort_values(by='spearman_corr', key=np.abs, ascending=False)

# הדפסת 20 הזוגות הקיצוניים ביותר שנמצאו במחקר
print(high_corr_df.head(20))

In [ ]:
# 14
'''
תא זה מציג ויזואליזציה של הקשרים החזקים ביותר שנמצאו בשלב הקודם.
מטרת הגרפים היא לאמת בעין את המתאם (Correlation) ולזהות את דפוסי ההתנהגות של הנתונים.
'''

# לולאה שעוברת על חמשת זוגות הפיצ'רים בעלי המתאם הגבוה ביותר שנמצאו קודם לכן
for _, row in high_corr_df.head(5).iterrows():
    # חילוץ שמות שני הפיצ'רים מתוך השורה הנוכחית בטבלה לצורך הצגה על הצירים
    a, b = row['feature_1'], row['feature_2']
    
    # יצירת גרף פיזור (Scatter Plot) המציג את הקשר בין המשתנה הראשון לשני
    # הפרמטר alpha=0.3 קובע שקיפות חלקית, המאפשרת לזהות הצטברות של נקודות (צפיפות נתונים)
    plt.scatter(df[a], df[b], alpha=0.3)
    
    # הגדרת שם הפיצ'ר הראשון כתווית לציר ה-X
    plt.xlabel(a)
    # הגדרת שם הפיצ'ר השני כתווית לציר ה-Y
    plt.ylabel(b)
    
    # הוספת כותרת לגרף המציגה את ערך מתאם ספירמן שחושב עבור זוג זה
    # הערך מעוגל לשתי ספרות אחרי הנקודה (:.2f) לצורך קריאות
    plt.title(f"Spearman={row['spearman_corr']:.2f}")
    
    # הצגת הגרף המוגמר על המסך לפני המעבר לזוג הבא ברשימה
    plt.show()

In [ ]:
# 15
'''
התא הנוכחי בוחן את הקורלציות (הקשרים) בין הפיצ'רים בתוך כל דרגת איכות (QoE Class) בנפרד.
המטרה המחקרית: לבדוק האם המאפיינים הרשתיים משנים את אופי הקשר ביניהם כתלות באיכות החוויה.
למשל: האם במצב של "פינג גרוע" (Class 3) הפיצ'רים מתנהגים בצורה שונה מאשר במצב "תקין".
'''

print("\n=== Correlation by classes ===")
# וידוא ניקוי שורות ללא ערכי פינג להבטחת שלמות הנתונים
df = df.dropna(subset=['ping'])
# יצירת עמודת הדירוג (1, 2, 3) המייצגת את רמות ה-QoE על בסיס פונקציות הדירוג שהגדרנו בשלב מוקדם
df['qoe_score'] = df['ping'].apply(lambda p: rank_str_to_int_mapping_3_classes.get(rank_qoe_3_classes(p), 0))

# מעבר בלולאה על כל אחת מדרגות האיכות הקיימות בבסיס הנתונים (בסדר עולה)
for label in sorted(df['qoe_score'].unique()):
    # סינון הנתונים כך שיכילו רק את הדגימות השייכות למחלקה (Label) הנוכחית
    sub = df[df['qoe_score'] == label]
    
    # חישוב מטריצת קורלציית ספירמן עבור הפיצ'רים בתוך תת-הקבוצה הספציפית
    corr = sub[FEATURE_COLUMNS].corr(method='spearman')
    
    print(f"\n=== Class {label} ===")
    # חישוב ממוצע הקורלציה של כל פיצ'ר ביחס לכל שאר הפיצ'רים באותה מחלקה.
    # המטרה: לזהות את חמשת הפיצ'רים ה"דומיננטיים" ביותר (אלו שיש להם את הקשר החזק ביותר עם שאר המדדים) באותה רמת איכות.
    print(corr.mean().sort_values(ascending=False).head(5))

In [ ]:
# 16
'''
השלב הנוכחי מבצע חישוב של מתאם ספירמן (Spearman Correlation) בין כל המשתנים לבין עמודת המטרה (qoe_score).
זהו שלב ה-Feature Importance הוויזואלי שלנו: אנו בודקים אילו מדדי רשת "רומזים" הכי חזק על הפינג ועל איכות החוויה.
'''

# חישוב מטריצת הקורלציה ובחירה ספציפית של הקשרים מול עמודת ה-qoe_score בלבד.
# מיון התוצאות מהגבוה לנמוך (ascending=False) מאפשר לראות מיד אילו פיצ'רים נמצאים במתאם חיובי או שלילי חזק עם הדירוג.

# Michael: הוספת numeric_only=True כדי למנוע קריסה על עמודות טקסטואליות (כמו folder)
df.corr(method='spearman', numeric_only=True)['qoe_score'].sort_values(ascending=False)

In [ ]:
# 17
'''
השלב הנוכחי מבצע השוואה של "דפוסי הקישוריות" בין הפיצ'רים השונים בכל מחלקת איכות בנפרד.
המטרה: לאתר את הפיצ'רים שמתנהגים הכי "אחרת" כאשר הרשת יציבה לעומת מצבי עומס או שיהוי.
זהו כלי עוצמתי להוכחת ההשערה שהרשת משנה את פניה כתלות ב-QoE.
'''

# יצירת מילון שישמור את ערכי הקישוריות הממוצעים לכל מחלקה (1, 2, 3)
per_class_top = {}

# לולאה שעוברת על כל אחת מדרגות האיכות הייחודיות בנתונים שלנו
for cls in sorted(df['qoe_score'].unique()):
    # סינון הנתונים לתת-קבוצה המכילה רק את הדגימות של המחלקה הנוכחית
    sub = df[df['qoe_score'] == cls]
    
    # חישוב מטריצת קורלציית ספירמן פנימית בתוך המחלקה
    # Michael: הוספת numeric_only=True כדי למנוע קריסה על עמודות שאינן מספרים
    corr = sub[FEATURE_COLUMNS].corr(method='spearman', numeric_only=True)
    
    # חישוב הממוצע של הערכים המוחלטים של הקורלציה לכל פיצ'ר.
    # המדד מראה עד כמה כל פיצ'ר "קשור" או "מסונכרן" עם שאר המדדים באותה רמת איכות.
    per_class_top[cls] = corr.abs().mean()

# ריכוז כל התוצאות לטבלת השוואה אחת (DataFrame)
df_compare = pd.DataFrame(per_class_top)

# חישוב ה-'range' (הטווח): ההפרש בין הערך המקסימלי למינימלי של הקישוריות לאורך כל המחלקות.
# פיצ'ר עם טווח גבוה הוא פיצ'ר שמשנה את אופיו בצורה משמעותית בין איכות טובה לגרועה.
df_compare['range'] = df_compare.max(axis=1) - df_compare.min(axis=1)

# הצגת 10 הפיצ'רים שדפוס ההתנהגות שלהם הוא הכי תלוי-איכות (בעלי הטווח הגדול ביותר).
df_compare.sort_values('range', ascending=False).head(10)
#זה מראה שדפוסי ההתנהגות של הרשת משתנים בין איכות חוויה טובה, בינונית וגרועה.

<a id="signatures"></a>
## 📈 Step 8: Network Signatures (Distribution & KDE)
Final visualization of how each network feature separates the QoE classes. Distinct curves indicate high predictive power.

In [ ]:
# 18
'''
התא הנוכחי מייצר גרפי התפלגות (היסטוגרמות) עבור כל אחד ממדדי הרשת (Features) שהגדרנו.
המטרה המחקרית: לזהות אילו פיצ'רים "מפרידים" טוב יותר בין דרגות האיכות. 
פיצ'ר שבו רואים הפרדה ברורה בין העקומות של QoE 1, 2 ו-3 הוא פיצ'ר בעל ערך חיזוי גבוה למודל.
'''

# מעבר בלולאה על כל אחד מהפיצ'רים ברשימת ה-FEATURE_COLUMNS שהגדרנו בשלב קודם
for col in FEATURE_COLUMNS:
    # פתיחת חלון ציור חדש עבור כל פיצ'ר בנפרד
    plt.figure()

    # הכנת הדאטה הכללי: טיפול בערכים חריגים (אינסוף) והסרת נתונים חסרים (NaN) למניעת שגיאות בגרף
    data_all = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    # ציור ההיסטוגרמה הכללית של כלל הדאטה כרקע (בצבע חלש, alpha=0.3)
    # density=True משמש לנרמול העקומות, כך שנוכל להשוות בין קבוצות בגדלים שונים
    plt.hist(data_all, bins=40, alpha=0.3, label="All QoE", density=True)

    # מעבר בלולאה פנימית על כל דרגת איכות בנפרד (1, 2, 3) כדי לראות את ה"חתימה" שלה
    for cls in sorted(df['qoe_score'].unique()):
        # סינון הנתונים עבור מחלקת האיכות הספציפית וניקוי ערכים לא תקינים
        data = df.loc[df['qoe_score'] == cls, col].replace([np.inf, -np.inf], np.nan).dropna()
        # ציור ההיסטוגרמה הייחודית של המחלקה
        plt.hist(data, bins=40, alpha=0.5, label=f"QoE {cls}", density=True)

    # הוספת שם הפיצ'ר ככותרת לגרף
    plt.title(col)
    # הוספת מקרא המציין איזו עקומה שייכת לאיזו דרגת איכות
    plt.legend()
    # הצגת הגרף המוגמר
    plt.show()

In [ ]:
# 19
'''
תא זה מפיק גרפי צפיפות (KDE Plots) עבור כל פיצ'ר, מחולקים לפי דרגות איכות ה-QoE.
היתרון של KDE על פני היסטוגרמה הוא היכולת לראות את ה"צורה" הסטטיסטית של הנתונים ללא השפעת גודל ה-Bins.
זהו המבחן הסופי להבנת הפוטנציאל של כל פיצ'ר להוות "סימן זיהוי" (Signature) לאיכות הרשת.
'''

# מעבר בלולאה על כל הפיצ'רים ברשימה שהגדרנו לצורך ניתוח מעמיק
for col in FEATURE_COLUMNS:
    # יצירת משטח ציור חדש לכל תכונה
    plt.figure()
    
    # מעבר על דרגות האיכות השונות (1, 2, 3) כדי להשוות את עקומות הצפיפות שלהן
    for cls in sorted(df['qoe_score'].unique()):
        # יצירת עקומת צפיפות חלקה (Kernel Density Estimate)
        sns.kdeplot(
            # סינון הנתונים למחלקה הנוכחית והסרת ערכים חסרים למניעת עיוותים בעקומה
            df[df['qoe_score'] == cls][col].dropna(),
            # הוספת תווית שתופיע במקרא (למשל: QoE 1)
            label=f"QoE {cls}",
            # מילוי השטח שמתחת לעקומה בצבע כדי להקל על זיהוי החפיפה בין הקבוצות
            fill=True
        )
    
    # הוספת שם הפיצ'ר ככותרת לגרף לצורך זיהוי מהיר של המדד הרשתי
    plt.title(col)
    # הצגת מקרא הצבעים להבחנה בין רמות האיכות
    plt.legend()
    # הצגת הגרף הסופי של המדד על המסך
    plt.show()

In [ ]:
# 20
# --- Michael: Final Notebook Summary ---
notebook_end_time = time.time()
total_duration = notebook_end_time - notebook_start_time

# שימוש ב-datetime כדי להציג את השעה הנוכחית בפורמט קריא
current_time = datetime.now().strftime('%H:%M:%S')

print(f"🏁 Notebook Visualization Completed in: {total_duration/60:.2f} minutes at {current_time}")

## 💾 Step 9: Final Persistence
Exporting the processed feature importance results for comparative study and archiving.

In [ ]:
# 21
'''
תא זה מבצע שמירה פיזית של תוצאות ניתוח חשיבות התכונות (Feature Importance) לכונן.
השמירה מאפשרת לנו לגשת לנתונים המעובדים גם ללא הרצה מחדש של המודלים,
מה שחוסך זמן רב בתהליך הניתוח המחקרי.
'''

# ייצוא הטבלה שיצרנו בשלב הקודם (imp_df) לקובץ CSV.
# אנו משתמשים בנתיב PRODUCTS_DIR שהוגדר בתחילת המחברת כדי לשמור על סדר בתיקיות הפרויקט.
# index=False מונע הוספה של עמודת אינדקס מיותרת לקובץ, מה ששומר עליו נקי לקריאה באקסל.
imp_df.to_csv(PRODUCTS_DIR / IMPORTANCE_FILENAME , index=False)